## Einführung 

Diese Lektion behandelt: 
- Was Funktionsaufrufe sind und wofür sie verwendet werden 
- Wie man einen Funktionsaufruf mit Azure OpenAI erstellt 
- Wie man einen Funktionsaufruf in eine Anwendung integriert 

## Lernziele 

Nach Abschluss dieser Lektion wissen Sie, wie und verstehen: 

- Den Zweck der Verwendung von Funktionsaufrufen 
- Einrichten eines Funktionsaufrufs mit dem Azure OpenAI-Dienst 
- Effektive Funktionsaufrufe für den Anwendungsfall Ihrer Anwendung zu entwerfen 


## Verständnis von Funktionsaufrufen 

Für diese Lektion möchten wir eine Funktion für unser Bildungs-Startup entwickeln, die es den Nutzern ermöglicht, einen Chatbot zu verwenden, um technische Kurse zu finden. Wir werden Kurse empfehlen, die ihrem Fähigkeitsniveau, ihrer aktuellen Rolle und ihrer Interessentechnologie entsprechen. 

Um dies zu erreichen, werden wir eine Kombination aus folgenden Komponenten verwenden: 
 - `Azure Open AI`, um dem Benutzer ein Chat-Erlebnis zu bieten
 - `Microsoft Learn Catalog API`, um den Benutzern zu helfen, Kurse basierend auf deren Anfragen zu finden 
 - `Function Calling`, um die Anfrage des Benutzers zu übernehmen und an eine Funktion zu senden, die die API-Anfrage stellt. 

Um zu beginnen, schauen wir uns an, warum wir Funktionsaufrufe überhaupt verwenden möchten: 


### Warum Funktionsaufrufe  

Wenn Sie bereits eine andere Lektion in diesem Kurs abgeschlossen haben, verstehen Sie wahrscheinlich die Leistungsfähigkeit von Large Language Models (LLMs). Hoffentlich können Sie auch einige ihrer Einschränkungen erkennen.  

Funktionsaufrufe sind eine Funktion des Azure Open AI Service, um die folgenden Einschränkungen zu überwinden:  
1) Konsistentes Antwortformat  
2) Möglichkeit, Daten aus anderen Quellen einer Anwendung im Chat-Kontext zu verwenden  

Vor Funktionsaufrufen waren die Antworten eines LLM unstrukturiert und inkonsistent. Entwickler mussten komplexen Validierungscode schreiben, um sicherzustellen, dass sie mit jeder Varianten einer Antwort umgehen können.  

Benutzer konnten keine Antworten wie „Wie ist das aktuelle Wetter in Stockholm?“ erhalten. Dies liegt daran, dass Modelle auf die Zeit beschränkt waren, zu der die Daten trainiert wurden.  

Schauen wir uns das folgende Beispiel an, das dieses Problem veranschaulicht:  

Nehmen wir an, wir möchten eine Datenbank mit Studentendaten erstellen, damit wir ihnen den richtigen Kurs vorschlagen können. Unten haben wir zwei Beschreibungen von Studenten, die in den enthaltenen Daten sehr ähnlich sind.  


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Wir möchten dies an ein LLM senden, um die Daten zu analysieren. Dies kann später in unserer Anwendung verwendet werden, um es an eine API zu senden oder in einer Datenbank zu speichern.

Lassen Sie uns zwei identische Aufforderungen erstellen, in denen wir das LLM anweisen, welche Informationen uns interessieren:


Wir möchten dies an ein LLM senden, um die Teile zu analysieren, die für unser Produkt wichtig sind. So können wir zwei identische Eingabeaufforderungen erstellen, um das LLM anzuleiten:


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Nach der Erstellung dieser beiden Eingabeaufforderungen senden wir sie mit `client.responses.create` an das LLM. Wir speichern die Eingabeaufforderung in der Variablen `input` und weisen die Rolle `user` zu. Dies dient dazu, eine Nachricht eines Benutzers nachzubilden, die an einen Chatbot gesendet wird. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Nun können wir beide Anfragen an das LLM senden und die Antwort, die wir erhalten, prüfen. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Obwohl die Eingabeaufforderungen gleich sind und die Beschreibungen ähnlich sind, können wir unterschiedliche Formate der Eigenschaft `Grades` erhalten. 

Wenn Sie die obige Zelle mehrfach ausführen, kann das Format entweder `3.7` oder `3.7 GPA` sein. 

Dies liegt daran, dass das LLM unstrukturierte Daten in Form der geschriebenen Eingabeaufforderung übernimmt und auch unstrukturierte Daten zurückgibt. Wir benötigen ein strukturiertes Format, damit wir wissen, was wir beim Speichern oder Verwenden dieser Daten erwarten können.

Durch den Einsatz der funktionalen Aufrufe können wir sicherstellen, dass wir strukturierte Daten zurückerhalten. Bei der Verwendung von Funktionsaufrufen ruft das LLM keine Funktionen tatsächlich auf oder führt sie aus. Stattdessen erstellen wir eine Struktur, der das LLM in seinen Antworten folgen soll. Diese strukturierten Antworten verwenden wir dann, um zu wissen, welche Funktion in unseren Anwendungen ausgeführt werden soll.  
 


![Funktionsaufruf-Flussdiagramm](../../../../translated_images/de/Function-Flow.083875364af4f4bb.webp)


Wir können dann das, was die Funktion zurückgibt, nehmen und an das LLM zurücksenden. Das LLM wird dann in natürlicher Sprache antworten, um die Anfrage des Benutzers zu beantworten. 


### Anwendungsfälle für den Einsatz von Funktionsaufrufen 

**Aufrufen externer Werkzeuge** 
Chatbots sind hervorragend darin, Antworten auf Fragen von Nutzern zu liefern. Durch die Verwendung von Funktionsaufrufen können die Chatbots Nachrichten von Nutzern verwenden, um bestimmte Aufgaben zu erledigen. Zum Beispiel kann ein Schüler den Chatbot bitten: „Sende eine E-Mail an meinen Lehrer, dass ich mehr Unterstützung zu diesem Thema benötige“. Dies kann einen Funktionsaufruf zu `send_email(to: string, body: string)` auslösen


**Erstellen von API- oder Datenbankabfragen**
Nutzer können Informationen mit natürlicher Sprache finden, die in eine formatierte Abfrage oder API-Anfrage umgewandelt wird. Ein Beispiel hierfür könnte ein Lehrer sein, der fragt „Wer sind die Schüler, die die letzte Aufgabe abgeschlossen haben“, was eine Funktion namens `get_completed(student_name: string, assignment: int, current_status: string)` aufrufen könnte


**Erstellen von strukturierten Daten**
Nutzer können einen Textblock oder CSV-Datei verwenden und das LLM nutzen, um wichtige Informationen daraus zu extrahieren. Zum Beispiel kann ein Schüler einen Wikipedia-Artikel über Friedensabkommen umwandeln, um KI-Lernkarten zu erstellen. Dies kann durch die Verwendung einer Funktion namens `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` erfolgen


## 2. Erstellen Ihres ersten Funktionsaufrufs 

Der Prozess zum Erstellen eines Funktionsaufrufs umfasst 3 Hauptschritte: 
1. Aufrufen der Chat Completions API mit einer Liste Ihrer Funktionen und einer Benutzeranfrage 
2. Lesen der Modellantwort, um eine Aktion auszuführen, z. B. eine Funktion oder einen API-Aufruf ausführen 
3. Einen weiteren Aufruf an die Chat Completions API machen mit der Antwort Ihrer Funktion, um diese Informationen zur Erstellung einer Antwort für den Benutzer zu verwenden. 


![Ablauf eines Funktionsaufrufs](../../../../translated_images/de/LLM-Flow.3285ed8caf4796d7.webp)


### Elemente eines Funktionsaufrufs

#### Benutzer Eingabe

Der erste Schritt besteht darin, eine Benutzernachricht zu erstellen. Diese kann dynamisch zugewiesen werden, indem der Wert einer Texteingabe genommen wird, oder Sie können hier einen Wert zuweisen. Wenn Sie zum ersten Mal mit der Chat Completions API arbeiten, müssen wir die `role` und den `content` der Nachricht definieren.

Die `role` kann entweder `system` (Regeln erstellen), `assistant` (das Modell) oder `user` (der Endbenutzer) sein. Für Funktionsaufrufe weisen wir dies als `user` und eine Beispiel-Frage zu.


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Funktionen erstellen. 

Als Nächstes definieren wir eine Funktion und die Parameter dieser Funktion. Wir verwenden hier nur eine Funktion namens `search_courses`, aber Sie können mehrere Funktionen erstellen.

**Wichtig**: Funktionen sind in der Systemnachricht an das LLM enthalten und zählen zur Anzahl der verfügbaren Tokens, die Ihnen zur Verfügung stehen. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definitionen** 

`name` - Der Name der Funktion, die wir aufrufen möchten. 

`description` - Dies ist die Beschreibung, wie die Funktion funktioniert. Hier ist es wichtig, spezifisch und klar zu sein 

`parameters` - Eine Liste von Werten und Formaten, die das Modell in seiner Antwort erzeugen soll 


`type` - Der Datentyp, in dem die Eigenschaften gespeichert werden 

`properties` - Liste der spezifischen Werte, die das Modell für seine Antwort verwendet 


`name` - der Name der Eigenschaft, die das Modell in seiner formatierten Antwort verwendet 

`type` - Der Datentyp dieser Eigenschaft 

`description` - Beschreibung der spezifischen Eigenschaft 


**Optional**

`required` - erforderliche Eigenschaft, damit der Funktionsaufruf abgeschlossen wird 


### Den Funktionsaufruf machen
Nachdem wir eine Funktion definiert haben, müssen wir sie nun in den Aufruf der Chat Completion API einbinden. Dies tun wir, indem wir `functions` zur Anfrage hinzufügen. In diesem Fall `functions=functions`.

Es gibt auch die Möglichkeit, `function_call` auf `auto` zu setzen. Das bedeutet, dass wir das LLM entscheiden lassen, welche Funktion basierend auf der Benutzernachricht aufgerufen werden soll, anstatt es selbst zuzuweisen.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Schauen wir uns nun die Antwort an und sehen, wie sie formatiert ist: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Sie sehen, dass der Name der Funktion aufgerufen wird und das LLM anhand der Benutzernachricht die Daten gefunden hat, um die Argumente der Funktion zu erfüllen. 


## 3. Integration von Funktionsaufrufen in eine Anwendung.


Nachdem wir die formatierte Antwort des LLM getestet haben, können wir sie nun in eine Anwendung integrieren.

### Steuerung des Ablaufs

Um dies in unsere Anwendung zu integrieren, gehen wir wie folgt vor:

Zuerst führen wir den Aufruf der Open AI-Dienste durch und speichern die Nachricht in einer Variablen namens `response_message`.


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Nun definieren wir die Funktion, die die Microsoft Learn API aufruft, um eine Liste von Kursen zu erhalten:


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Als bewährte Methode werden wir dann prüfen, ob das Modell eine Funktion aufrufen möchte. Danach erstellen wir eine der verfügbaren Funktionen und ordnen sie der aufgerufenen Funktion zu. 
Anschließend nehmen wir die Argumente der Funktion und ordnen sie den Argumenten des LLM zu.

Schließlich fügen wir die Funktionsaufrufnachricht und die Werte, die durch die `search_courses`-Nachricht zurückgegeben wurden, an. Dies gibt dem LLM alle Informationen, die es benötigt, um
mit natürlicher Sprache auf den Benutzer zu antworten. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Jetzt senden wir die aktualisierte Nachricht an das LLM, damit wir eine Antwort in natürlicher Sprache statt einer im API-JSON-Format erhalten können. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Code-Herausforderung

Großartige Arbeit! Um Ihr Lernen der Azure Open AI Funktionsaufrufe fortzusetzen, können Sie folgendes erstellen: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst
 - Weitere Parameter der Funktion, die Lernenden helfen könnten, mehr Kurse zu finden. Die verfügbaren API-Parameter finden Sie hier:
 - Erstellen Sie einen weiteren Funktionsaufruf, der mehr Informationen vom Lernenden abfragt, wie z.B. seine Muttersprache
 - Erstellen Sie eine Fehlerbehandlung, wenn der Funktionsaufruf und/oder API-Aufruf keine geeigneten Kurse zurückgibt


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
